# AgentCore Memory(장기 메모리)를 사용하는 Strands Agents - Built-in Strategies

## 소개

이 튜토리얼에서는 **built-in strategy**를 사용하는 **MemoryManager** 및 **MemorySessionManager**와 hook을 통해 AgentCore Memory에 통합된 Strands agent로 **지능형 고객 지원 에이전트**를 구축하는 방법을 살펴봅니다. 고객 상호 작용 기록을 장기 메모리에 보관하고 구매 세부 정보를 기억하여 이전 대화와 사용자 선호도를 바탕으로 개인화된 지원을 제공하는 데 중점을 둡니다.

**참고: 이 접근 방식은 IAM execution role이 필요하지 않은 built-in strategy(SemanticStrategy, UserPreferenceStrategy)를 사용합니다.**

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| Agent 유형          | 고객 지원                                                                         |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore Semantic 및 User Preferences Memory Extraction(Built-in), 메모리 저장 및 검색용 Hook |
| 예제 난이도         | 중급                                                                              |

다음 내용을 학습합니다.
- MemoryManager를 사용하여 built-in 장기 strategy로 AgentCore Memory 설정
- MemorySessionManager로 자동 저장 및 검색을 수행하는 memory hook 생성
- 지속형 메모리를 사용하는 고객 지원 에이전트 구축
- 이전 상호 작용의 맥락을 활용하여 고객 문제 처리
- IAM role 구성 없이 built-in strategy 사용

### 시나리오 배경
이 예제에서는 **고객 지원 사용 사례**를 구축합니다. 에이전트는 주문 기록, 선호도, 이전 문제를 포함한 고객 맥락을 기억하여 더 개인화되고 효과적인 지원을 제공합니다. 고객과의 대화는 memory hook을 통해 자동으로 저장되므로 중요한 세부 정보가 누락되지 않습니다. Semantic 및 User Preference 같은 여러 memory strategy를 사용하면 에이전트가 폭넓은 관련 정보를 포착할 수 있습니다. 이 구성을 통해 에이전트는 고객의 기록과 선호도를 충분히 파악한 상태에서 문제를 해결할 수 있습니다. 또한 web search 기능을 통합하여 필요에 따라 최신 제품 정보와 troubleshooting 지침을 쉽게 제공할 수 있습니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
- Python 3.10+
- Amazon Bedrock AgentCore Memory 권한이 있는 AWS credentials
- Amazon Bedrock AgentCore SDK with MemoryManager support

## 📊 Built-in Strategy와 Custom Strategy 선택

AgentCore Memory는 메모리 추출 및 통합을 위한 두 가지 접근 방식을 제공합니다. 이 Notebook에서는 **Built-in Strategies** 접근 방식을 살펴봅니다.

### Built-in Strategies (이 Notebook)

**사용 시점:**
- ✅ 빠른 설정 및 prototype 제작
- ✅ 표준 메모리 추출 요구 사항
- ✅ 사용자 지정 모델 선택이 필요하지 않은 경우
- ✅ 간소화된 IAM 구성(execution role 불필요)
- ✅ 기본 AgentCore model을 사용하는 production workload

**주요 특징:**
- `SemanticStrategy` 및 `UserPreferenceStrategy` class 사용
- AgentCore Memory가 model을 자동으로 선택하고 관리
- IAM execution role 불필요
- Parameter가 적어 구성이 간단함
- 대부분의 사용 사례에 적합

**예제:**
```python
strategies = [
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["/support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies
    # memory_execution_role_arn은 필요하지 않음
)
```

### Custom Strategy Override (`customer-support-override-strategy.ipynb` 참고)

**사용 시점:**
- ✅ 추출 및 통합용 사용자 지정 Bedrock model을 지정해야 하는 경우
- ✅ Model 동작을 세밀하게 제어해야 하는 경우
- ✅ 추출 및 통합용 사용자 지정 prompt가 필요한 경우
- ✅ 특정 model 기능이 필요한 고급 사용 사례
- ✅ 특정 model version에 대한 규정 준수 요구 사항

**주요 특징:**
- `CustomSemanticStrategy` 및 `CustomUserPreferenceStrategy` class 사용
- Model 호출용 IAM execution role 필요
- `ExtractionConfig` 및 `ConsolidationConfig` 지정 가능
- 설정은 더 복잡하지만 제어 범위가 넓음
- 전문적인 요구 사항에 유용

**예제:**
```python
strategies = [
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        extraction_config=ExtractionConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Extract factual information..."
        ),
        consolidation_config=ConsolidationConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Consolidate semantic insights..."
        ),
        namespaces=["/support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies,
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN  # 필수
)
```

### 빠른 비교표

| 기능 | Built-in Strategies | Custom Strategy Override |
|---------|---------------------|-------------------------|
| **설정 복잡도** | 간단 | 고급 |
| **IAM Role 필요 여부** | ❌ 아니요 | ✅ 예 |
| **Model 선택** | 자동(AgentCore 관리) | 수동(사용자 지정) |
| **Custom Prompt** | ❌ 아니요 | ✅ 예 |
| **구성** | 최소 | 상세 |
| **사용 사례** | 표준 메모리 추출 | Custom model 요구 사항 |
| **권장 대상** | 대부분의 application | 전문적인 요구 사항 |

---

**💡 권장 사항:** 대부분의 사용 사례에서는 built-in strategy(이 Notebook)로 시작하세요. 특정 model 요구 사항이 있거나 추출 및 통합 동작을 세밀하게 제어해야 할 때만 custom strategy override를 사용하세요.

## 1단계: Dependency 설치 및 설정
이 Notebook 실행에 필요한 모든 library를 import하고 client를 정의하겠습니다.

In [1]:
!pip install -qr requirements.txt

In [2]:
import logging
from typing import List
from datetime import datetime

# Logging 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Strands Agent에 필요한 module import
from strands import Agent, tool
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)
from ddgs import DDGS

# Memory 관리 module import
from bedrock_agentcore_starter_toolkit.operations.memory.manager import (
    MemoryManager,
)
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy,
    UserPreferenceStrategy,
)
from bedrock_agentcore.memory.constants import (
    ConversationalMessage,
    MessageRole,
    RetrievalConfig,
)
from bedrock_agentcore.memory.models import (
    StringValue,
    MemoryRecord,
)
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Message role 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

logger.info("✅ All imports loaded successfully")

In [3]:
# 구성 - 올바른 값으로 교체
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info("✅ Configuration loaded")
logger.info(f"   Region: {REGION}")
logger.info(f"   Customer ID: {CUSTOMER_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## 2단계: 고객 지원용 Memory Resource 생성

고객 지원을 위해 여러 built-in memory strategy를 사용합니다.
- **UserPreferenceStrategy**: 고객 선호도와 행동 포착(built-in)
- **SemanticStrategy**: 주문 fact와 제품 정보 저장(built-in)

**중요**: Built-in strategy에는 IAM execution role이 필요하지 않습니다. AgentCore Memory는 추출 및 통합에 기본 model을 사용합니다.

In [ ]:
# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "CustomerSupportLongTermMemory"

# 기본 memory manager 초기화 및 연결 테스트
logger.info(f"✅ MemoryManager initialized for region: {REGION}")
logger.info(f"Memory manager type: {type(memory_manager)}")

# Built-in strategy class로 memory strategy 정의
strategies = [
    UserPreferenceStrategy(
        name="CustomerPreferences",
        description="Captures customer preferences and behavior",
        namespaces=["/support/customer/{actorId}/preferences/"],
    ),
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["/support/customer/{actorId}/semantic/"],
    ),
]

# Strategy 구성 검증
logger.info(f"✅ Configured {len(strategies)} built-in memory strategies:")
for i, strategy in enumerate(strategies, 1):
    logger.info(f"  {i}. {strategy.name} ({type(strategy).__name__})")
    logger.info(f"     Description: {strategy.description}")
    logger.info(f"     Namespaces: {strategy.namespaces}")
    logger.info("     Uses default AgentCore models (no IAM role required)")

# Built-in strategy와 MemoryManager를 사용하여 memory resource 생성
logger.info(f"Creating memory '{memory_name}' with {len(strategies)} built-in strategies...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,  # Built-in strategy object 전달
        description="Memory for customer support agent with built-in strategies",
        event_expiry_days=90,  # Memory는 90일 후 만료됨
        # 참고: Built-in strategy에는 memory_execution_role_arn이 필요하지 않음
    )
    memory_id = memory.id
    logger.info("✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
    logger.info("   Using built-in strategies (no IAM role required)")

except Exception as e:
    # 향상된 오류 보고로 memory 생성 중 발생하는 오류 처리
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 정리 - memory가 일부 생성되었다면 삭제
    if "memory_id" in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")

    # 원래 exception을 다시 발생시킴
    raise

In [ ]:
# Memory manager 기본 기능 테스트
try:
    # 기존 memory 나열 테스트
    existing_memories = memory_manager.list_memories()
    logger.info(f"✅ Memory manager connection successful. Found {len(existing_memories)} existing memories")

    # 기존 test memory 정리
    for mem in existing_memories:
        if mem.name and mem.name.startswith("CustomerSupportLongTermMemory"):
            logger.info(f"Cleaning up existing test memory: {mem.id}")
            memory_manager.delete_memory(mem.id)

except Exception as e:
    logger.error(f"❌ Memory manager test failed: {e}")
    raise

Memory에 지정한 strategy가 포함되어 있는지 확인해 보겠습니다.

In [ ]:
# MemoryManager로 memory 정보 표시
print(f"Memory ID: {memory.id}")
print(f"Memory Name: {memory.name}")
print(f"Memory Description: {memory.description}")
print(f"Memory Status: {memory.status}")
print(f"Number of strategies: {len(strategies)}")
for i, strategy in enumerate(strategies, 1):
    print(f"  {i}. {strategy.name}: {strategy.description}")

## 3단계: Agent Tool 생성

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException


@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for product information, troubleshooting guides, or support articles.

    Args:
        query: Search query for product info or troubleshooting
        max_results: Maximum number of results to return

    Returns:
        Search results with titles and snippets
    """
    try:
        results = DDGS().text(query, region="us-en", max_results=max_results)
        if not results:
            return "No search results found."

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n   {result.get('body', 'No description')}"
            )

        return "\n".join(formatted_results)
    except RatelimitException:
        return "Rate limit reached: Please try again after a short delay."
    except DDGSException as d:
        return f"Search Error: {d}"
    except Exception as e:
        return f"Search error: {str(e)}"


logger.info("✅ Web search tool ready")


@tool
def check_order_status(order_number: str) -> str:
    """Check the status of a customer order.

    Args:
        order_number: The order number to check

    Returns:
        Order status information
    """
    # 주문 조회 시뮬레이션
    mock_orders = {
        "123456": "iPhone 15 Pro - Delivered on June 5, 2025",
        "654321": "Sennheiser Headphones - Delivered on June 25, 2025, 1-year warranty active",
        "789012": "Samsung Galaxy S23 - In transit, expected delivery on July 1, 2025",
    }

    return mock_orders.get(order_number, f"Order {order_number} not found. Please verify the order number.")


logger.info("✅ Check Order Status tool ready")

## 4단계: Session Manager 초기화

**새 기능: 이 섹션에서는 session 기반 Memory 작업을 위한 MemorySessionManager를 소개합니다.**

In [ ]:
# Session memory manager 초기화
session_manager: MemorySessionManager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# 특정 고객용 memory session 생성
customer_session: MemorySession = session_manager.create_memory_session(actor_id=CUSTOMER_ID, session_id=SESSION_ID)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"✅ Customer session created for actor: {CUSTOMER_ID}")
logger.info(f"   Session type: {type(customer_session)}")
logger.info(f"   Actor object: {customer_session.get_actor()}")

## 5단계: 고객 지원용 Memory Hook Provider 생성
Hook은 에이전트 실행 lifecycle의 특정 시점에 실행되는 특수 function입니다. 사용자 지정 hook provider는 다음 방식으로 고객 지원 맥락을 자동 관리합니다.
- Session 기반 method를 사용하여 각 응답 후 **지원 상호 작용 저장**
- 새 질의를 처리할 때 이전 주문과 선호도에서 **관련 맥락을 검색하여 주입**


In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    """MemorySession으로 개선된 고객 지원 에이전트용 메모리 훅입니다."""

    def __init__(self, customer_session: MemorySession):
        # MemorySession을 직접 받음
        self.customer_session = customer_session

        # 서로 다른 memory type의 검색 구성 정의
        self.retrieval_config = {
            "/support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
            "/support/customer/{actorId}/semantic/": RetrievalConfig(top_k=5, relevance_score=0.2),
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """지원 질의를 처리하기 전에 MemorySession으로 고객 컨텍스트를 검색합니다."""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]

            try:
                # 맥락 검색에 MemorySession 사용
                relevant_memories = []

                # MemorySession으로 여러 memory namespace 검색
                for namespace_template, config in self.retrieval_config.items():
                    # Session의 실제 actor ID로 namespace template 해석
                    resolved_namespace = namespace_template.format(actorId=self.customer_session._actor_id)

                    # MemorySession API 사용 (actor_id/session_id 전달 불필요)
                    memories = self.customer_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k,
                    )

                    # 관련성 score로 필터링
                    filtered_memories = [
                        memory for memory in memories if memory.get("score", 0) >= config.relevance_score
                    ]

                    relevant_memories.extend(filtered_memories)
                    logger.info(
                        f"Found {len(filtered_memories)} relevant memories in {resolved_namespace} (filtered from {len(memories)} total)"
                    )

                # Memory를 찾으면 에이전트의 system prompt에 맥락 주입
                if relevant_memories:
                    context_text = self._format_context(relevant_memories)
                    original_prompt = event.agent.system_prompt
                    enhanced_prompt = f"{original_prompt}\n\nCustomer Context:\n{context_text}"
                    event.agent.system_prompt = enhanced_prompt
                    logger.info(f"✅ Injected {len(relevant_memories)} memories into agent context")

            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")

    def _format_context(self, memories: List[MemoryRecord]) -> str:
        """검색한 메모리를 에이전트 컨텍스트 형식으로 변환합니다."""
        context_lines = []
        for i, memory in enumerate(memories[:5], 1):  # 상위 5개로 제한
            content = memory.get("content", {}).get("text", "No content available")
            score = memory.get("score", 0)
            context_lines.append(f"{i}. (Score: {score:.2f}) {content[:200]}...")

        return "\n".join(context_lines)

    def save_support_interaction(self, event: AfterInvocationEvent):
        """MemorySession의 간결한 API로 지원 상호 작용을 저장합니다."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 고객 질의와 에이전트 응답 가져오기
                customer_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break

                if customer_query and agent_response:
                    # MemorySession 사용 (actor_id/session_id 전달 불필요)
                    interaction_messages = [
                        ConversationalMessage(customer_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT),
                    ]

                    result = self.customer_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved interaction using MemorySession - Event ID: {result['eventId']}")

        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """고객 지원 메모리 훅을 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)  # 다시 추가됨
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("✅ Customer support memory hooks registered with MemorySession")


print("Executed!")

### 6단계: 고객 지원 Agent 생성

In [ ]:
# MemorySession을 사용하여 memory hook 생성
support_hooks = CustomerSupportMemoryHooks(customer_session)

# 고객 지원 에이전트 생성
support_agent = Agent(
    hooks=[support_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[web_search, check_order_status],
    system_prompt="""You are a helpful customer support agent with access to customer history and order information. 
    
    Your role:
    - Help customers with their orders, returns, and product issues
    - Use customer context to provide personalized support
    - Search for product information when needed
    - Be empathetic and solution-focused
    - Reference previous orders and preferences when relevant
    
    Always be professional, helpful, and aim to resolve customer issues efficiently.""",
)

print("✅ Customer support agent created with MemorySession integration")

### 7단계: 고객 기록 Seed

메모리 기능을 시연하도록 이전 고객 상호 작용을 추가하겠습니다.

**참고: 이 섹션에서는 ConversationalMessage 형식과 session 기반 저장을 사용합니다.**

In [ ]:
# MemorySession으로 이전 고객 상호 작용 seed
previous_interactions = [
    ConversationalMessage("I bought a new iPhone 15 Pro on June 1st, 2025. Order number is 123456.", USER),
    ConversationalMessage(
        "Thank you for your purchase! I can see your iPhone 15 Pro order #123456 was delivered successfully. How can I help you today?",
        ASSISTANT,
    ),
    ConversationalMessage(
        "I also ordered Sennheiser headphones on June 20th. Order number 654321. They came with 1-year warranty.",
        USER,
    ),
    ConversationalMessage(
        "Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty. Both your iPhone and headphones should work great together.",
        ASSISTANT,
    ),
    ConversationalMessage("I'm looking for a good laptop. I prefer ThinkPad models.", USER),
    ConversationalMessage(
        "Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.",
        ASSISTANT,
    ),
]

# MemorySession으로 저장
try:
    event_response = customer_session.add_turns(previous_interactions)
    logger.info("✅ Seeded customer history using MemorySession")
    logger.info(f"   Event ID: {event_response['eventId']}")
except Exception as e:
    logger.error(f"⚠️ Error seeding history: {e}")

#### Agent를 사용할 준비가 되었습니다.

### 고객 지원 시나리오 테스트

In [ ]:
# 테스트 1: 고객이 iPhone 문제를 보고
logger.info("🧪 Running Test 1: iPhone performance issue")
test_query_1 = "My iPhone is running very slow and gets hot when charging. Can you help?"
logger.info(f"Query: {test_query_1}")

response1 = support_agent(test_query_1)
logger.info("✅ Test 1 completed successfully")
print(f"\n📱 iPhone Issue Support Response:\n{response1}\n")

In [ ]:
# 테스트 2: Bluetooth 연결 문제
logger.info("🧪 Running Test 2: Bluetooth connectivity issue")
test_query_2 = "My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?"
logger.info(f"Query: {test_query_2}")

response2 = support_agent(test_query_2)
logger.info("✅ Test 2 completed successfully")
print(f"\n🎧 Bluetooth Issue Support Response:\n{response2}\n")

In [ ]:
# 테스트 3: 주문 상태 확인
logger.info("🧪 Running Test 3: Order status check")
test_query_3 = "Can you check the status of my recent orders?"
logger.info(f"Query: {test_query_3}")

response3 = support_agent(test_query_3)
logger.info("✅ Test 3 completed successfully")
print(f"\n📦 Order Status Support Response:\n{response3}\n")

In [ ]:
# 테스트 4: 선호도 기반 제품 추천
logger.info("🧪 Running Test 4: Product recommendation")
test_query_4 = "I'm still interested in buying a laptop. What ThinkPad models do you recommend?"
logger.info(f"Query: {test_query_4}")

response4 = support_agent(test_query_4)
logger.info("✅ Test 4 completed successfully")
print(f"\n💻 Product Recommendation Support Response:\n{response4}\n")

logger.info("🎉 All customer support scenario tests completed!")

## 고급 기능: Branching 및 Metadata

### SessionManager를 사용한 대화 Branching

Branching을 사용하여 다른 지원 시나리오를 탐색합니다.

In [ ]:
# 대화의 마지막 event ID 가져오기
events = customer_session.list_events()
if events:
    last_event_id = events[-1].eventId

    # Premium 지원 경로를 탐색하도록 대화 fork
    branch_event = customer_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="premium-support",
        messages=[
            ConversationalMessage("I'd like to upgrade to premium support for faster resolution.", USER),
            ConversationalMessage(
                "Excellent choice! With premium support, you'll get 24/7 priority assistance, dedicated account manager, and same-day resolution guarantee. Let me process your upgrade.",
                ASSISTANT,
            ),
        ],
    )

    logger.info(f"✅ Created premium support branch from event {last_event_id}")

    # 모든 branch 나열
    branches = customer_session.list_branches()
    print(f"\n🌳 Support session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")
else:
    print("No events found to branch from")

### 고급 지원 추적용 Metadata

Metadata를 사용하여 종합적인 지원 metric을 추적합니다.

In [ ]:
# 종합적인 metadata가 포함된 지원 상호 작용 추가
metadata_event = customer_session.add_turns(
    messages=[
        ConversationalMessage(
            "The ThinkPad X1 Carbon you recommended is perfect! I'll order it now.",
            USER,
        ),
        ConversationalMessage(
            "Fantastic! The ThinkPad X1 Carbon is an excellent choice for your needs. I'll help you complete the order with your preferred configuration.",
            ASSISTANT,
        ),
    ],
    metadata={
        "interaction_type": StringValue.build("product_recommendation"),
        "outcome": StringValue.build("purchase_intent"),
        "product_category": StringValue.build("laptops"),
        "product_brand": StringValue.build("lenovo"),
        "customer_sentiment": StringValue.build("positive"),
        "support_tier": StringValue.build("standard"),
        "session_duration_minutes": StringValue.build("15"),
    },
)

logger.info(f"✅ Added support event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Support interaction tagged with:")
print("   - Interaction Type: product_recommendation")
print("   - Outcome: purchase_intent")
print("   - Product Category: laptops")
print("   - Customer Sentiment: positive")

### 고급 Metadata 질의

지원 pattern과 고객 행동을 분석합니다.

In [ ]:
try:
    # 제품 추천 상호 작용 질의
    recommendation_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "interaction_type"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "product_recommendation"}},
            }
        ]
    )

    print(f"\n🛍️ Found {len(recommendation_events)} product recommendation interaction(s)")

    # 긍정적 sentiment 상호 작용 질의
    positive_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "customer_sentiment"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "positive"}},
            }
        ]
    )

    print(f"😊 Found {len(positive_events)} positive sentiment interaction(s)")

    print("\n💡 Advanced analytics use cases:")
    print("   - Track conversion rates from recommendations to purchases")
    print("   - Analyze customer sentiment trends over time")
    print("   - Identify most effective support interaction types")
    print("   - Measure support tier performance")
    print("   - Generate detailed customer journey reports")
    print("   - Optimize product recommendation strategies")

except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print("Note: Metadata filtering requires events with metadata tags")

#### 고객 지원 튜토리얼을 완료했습니다! 🎉
핵심 내용:
- Memory hook은 MemorySessionManager를 사용하여 여러 지원 session의 고객 맥락을 자동 관리합니다.
- Built-in strategy(SemanticStrategy, UserPreferenceStrategy)는 IAM role 없이 주문, 선호도, fact를 포착합니다.
- 에이전트는 고객 기록을 바탕으로 개인화된 지원을 제공할 수 있습니다.
- 주문 조회 및 web search 기능을 위한 tool을 통합할 수 있습니다.
- 지속형 메모리를 사용하면 고객 지원의 효율성이 높아집니다.
- **Branching을 통해 다른 지원 방식과 escalation 경로를 테스트할 수 있습니다.**
- **Metadata는 종합적인 지원 분석과 customer journey 추적을 제공합니다.**
- **Built-in strategy는 IAM role 구성을 없애 설정을 단순화합니다.**

## 정리

### 선택 사항: Memory Resource 삭제

In [ ]:
# MemoryManager로 memory resource를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     print(f"✅ Deleted memory resource using MemoryManager: {memory_id}")
# except Exception as e:
#     print(f"Error deleting memory: {e}")